# BM25 Baseline Analysis

Goal: understand BM25 ranking and find easy and difficult queries.

This notebook analyzes the BM25 baseline ranking. 

The idea is to understand how well BM25 retrieves relevant passages, identify easy and difficult queries, and inspect examples of retrieval successes and failures before applying DuoBERT and explanation methods.

In [1]:
## -- IMPORTS -- 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

In [2]:
# load queries
queries = pd.read_csv(
    "../data/msmarco_passage_dev/raw/queries.dev.tsv",
    sep="\t",
    header=None,
    names=["query_id", "query"])

queries.head()

,query_id,query
0,1048578,cost of endless pools/swim spa
1,1048579,what is pcnt
2,1048580,what is pcb waste
3,1048581,what is pbis?
4,1048582,what is paysky


In [3]:
# load qrels
qrels = pd.read_csv(
    "../data/msmarco_passage_dev/raw/qrels.dev.tsv",
    sep="\t",
    header=None,
    names=["query_id", "unused", "pid", "relevance"]).drop(columns=["unused"])

qrels.head()

,query_id,pid,relevance
0,1102432,2026790,1
1,1102431,7066866,1
2,1102431,7066867,1
3,1090282,7066900,1
4,39449,7066905,1


In [4]:
# load top1000
top1000 = pd.read_csv(
    "../data/msmarco_passage_dev/raw/top1000.dev",
    sep="\t",
    header=None,
    names=["query_id", "pid", "query", "passage"])

top1000.head()

,query_id,pid,query,passage
0,188714,1000052,foods and supplements to lower blood sugar,Watch portion sizes: ■ Even healthy foods will...
1,1082792,1000084,what does the golgi apparatus do to the protei...,"Start studying Bonding, Carbs, Proteins, Lipid..."
2,995526,1000094,where is the federal penitentiary in ind,It takes THOUSANDS of Macy's associates to bri...
3,199776,1000115,health benefits of eating vegetarian,The good news is that you will discover what g...
4,660957,1000115,what foods are good if you have gout?,The good news is that you will discover what g...


In [ ]:
# add rank to top1000
top1000["rank"] = top1000.groupby("query_id").cumcount() + 1

# add relevance labels to the ranking
candidates = top1000.merge(
    qrels[["query_id", "pid", "relevance"]],
    on=["query_id", "pid"],
    how="left")

candidates["relevance"] = candidates["relevance"].fillna(0)
candidates.head()

,query_id,pid,query,passage,rank,relevance
0,188714,1000052,foods and supplements to lower blood sugar,Watch portion sizes: ■ Even healthy foods will...,1,0.0
1,1082792,1000084,what does the golgi apparatus do to the protei...,"Start studying Bonding, Carbs, Proteins, Lipid...",1,0.0
2,995526,1000094,where is the federal penitentiary in ind,It takes THOUSANDS of Macy's associates to bri...,1,0.0
3,199776,1000115,health benefits of eating vegetarian,The good news is that you will discover what g...,1,0.0
4,660957,1000115,what foods are good if you have gout?,The good news is that you will discover what g...,1,0.0


In [ ]:
# for each query, find the rank of the first relevant passage
first_rel_rank = (candidates[candidates["relevance"] == 1].groupby("query_id")["rank"].min().reset_index())
first_rel_rank.columns = ["query_id", "first_relevant_rank"]
first_rel_rank.head()

,query_id,first_relevant_rank
0,2,936
1,1215,886
2,2235,219
3,2798,443
4,2962,634


In [ ]:
# summary statistics
first_rel_rank["first_relevant_rank"].describe()

count    5738.000000
mean      502.013245
std       299.547608
min         1.000000
25%       218.000000
50%       515.000000
75%       760.000000
max      1000.000000
Name: first_relevant_rank, dtype: float64

### Interpretation
- The average rank of the first relevant passage is approximately 502, with a median of 515 -> relevant passages are often located far down in the BM25 ranking
- This suggests that BM25 alone does not rank relevant passages highly, highlighting the need for a reranking model such as DuoBERT
- Additionally, not all queries have a relevant passage in the candidate set, confirming earlier observations about retrieval limitations

## Easy and hard queries

In [15]:
easy_queries = first_rel_rank[first_rel_rank["first_relevant_rank"] <= 10]
hard_queries = first_rel_rank[first_rel_rank["first_relevant_rank"] > 100]

print("Easy queries:", len(easy_queries))
print("Hard queries:", len(hard_queries))

Easy queries: 106
Hard queries: 4994


This shows that BM25 struggles for most queries, since only a very small number of queries have a relevant passage in the top 10.

In [19]:
def show_full_example(qid):
    q = queries[queries["query_id"] == qid]["query"].values[0]
    subset = candidates[candidates["query_id"] == qid].sort_values("rank")
    
    print("Query:", q)
    
    # top 5
    print("\nTop 5 passages:\n")
    for _, row in subset.head(5).iterrows():
        print(f"Rank {row['rank']} | Relevant: {row['relevance']}")
        print(row["passage"][:200])
        print("-" * 50)
    
    # first relevant
    rel = subset[subset["relevance"] == 1].iloc[0]
    
    print("\nFirst relevant passage:")
    print(f"Rank {rel['rank']}")
    print(rel["passage"][:200])

# Easy example
print("Easy Query Example:")
show_full_example(easy_queries.iloc[0]["query_id"])

print("\n" + "="*80 + "\n")

# Hard example
print("Hard Query Example:")
show_full_example(hard_queries.iloc[0]["query_id"])

Easy Query Example:
Query: cost to dig a pond

Top 5 passages:

Rank 1 | Relevant: 0.0
All water bodies, in order of size, it would be Pond, Lagoon, Lake, and River; the first three are stagnant, the last is flowing water. A pond is a body of standing water, either natural or artificial
--------------------------------------------------
Rank 2 | Relevant: 0.0
Wayne, Maine. 1  Wayne: On the road into wayne, Maine. 2  Wayne: wilson pond. 3  Wayne: Pocasset Lake, Wayne.  Wayne: wilson 1  pond. Wayne: wilson pond.  Wayne: Ducks at Wayne 1  Dam. Wayne: In the w
--------------------------------------------------
Rank 3 | Relevant: 0.0
Circular shaped pond: -measure the total distance around the pond surface in feet and multiply this number by itself and divide by 547,560. Rectangular shaped pond: -measure the length of one side and
--------------------------------------------------
Rank 4 | Relevant: 0.0
Excavate, a product offered by Sanco, is an all-natural biological pond dredging product

### Easy Query: “cost to dig a pond”

#### Observations:

The first relevant passage appears at rank 6, which means BM25 retrieves the correct information relatively early in the ranking.

The top-ranked passages (ranks 1–5) all contain the keyword “pond”, but they focus on:
* definitions of ponds
* locations
* measurement or volume calculations

None of the top-ranked passages address the cost aspect of the query. The relevant passage at rank 6 explicitly mentions price estimates (e.g., cost per cubic yard).

#### Interpretation:
- BM25 is able to retrieve relevant passages within the candidate set, indicating that keyword overlap is sufficient to find useful documents.
- However, it fails to rank the most relevant passage at the top because it does not distinguish between different aspects of the query (e.g., “pond” vs “cost”).
- This shows that BM25 prioritizes passages based on term frequency and overlap, rather than the importance of specific query terms.
- The query is therefore “easy” in the sense that the relevant passage is present and relatively high in the ranking, but BM25 still produces suboptimal ordering.

____________________________________________________________________
### Hard Query: “androgen receptor define”

#### Observations:

The first relevant passage appears at rank 936, indicating that BM25 ranks the correct answer extremely low.
The top-ranked passages (ranks 1–5) contain related terms such as “androgen” and “receptor”, but:
* they discuss general biological processes
* they mention related concepts rather than defining the term

The relevant passage at rank 936 provides a clear and direct definition of “androgen receptor”.

#### Interpretation:
- BM25 retrieves passages that contain overlapping keywords but does not capture the specific intent of the query, which is to obtain a definition.
- The model treats all occurrences of “androgen” and “receptor” similarly, without distinguishing between:
    - explanatory or contextual passages
    - precise definitional answers
- As a result, the truly relevant passage is buried deep in the ranking.
This demonstrates a clear limitation of BM25: it lacks semantic understanding and intent awareness.